# Edmonton International Flight Price Analysis

## End-to-end portfolio case study

This project began with a practical travel question:

> **Can historical Google Flights price data help identify when to book, and which international destinations are cheapest from Edmonton for a fixed travel period?**

I built the project in stages instead of starting with a finished dataset. This is important because real analytics work often changes direction after the first data is collected.

### Project evolution

1. Test whether Google Flights price history can be collected through SerpApi.
2. Analyze historical prices for **Edmonton (YEG) → Mexico City (MEX)**.
3. Repeat the idea for **YEG → Buenos Aires (EZE)**.
4. Broaden the question from *when should I buy?* to *where can I travel cheaply on fixed dates?*
5. Build a candidate universe of airports in Central America, South America, the Caribbean, and Iceland.
6. Reduce the very large airport list to **30 selected popular destinations**.
7. Search those destinations in batches for **November 23–December 5, 2026**.
8. Individually verify the 10 cheapest results.
9. Perform exploratory data analysis on the verified shortlist.

### Skills demonstrated

`Python` · `pandas` · `NumPy` · `Matplotlib` · `requests` · `JSON` · `APIs` · `data cleaning` · `feature engineering` · `EDA` · `data validation`

> **Important:** Airfares are dynamic. The saved files in this repository are snapshots from the searches performed during the project, not live booking quotes.

## 1. Import libraries and locate project files

The notebook uses saved CSV snapshots for the analysis so that it can be opened on GitHub without exposing an API key.

The optional data-collection sections show how the API searches were performed.

In [ ]:
import os
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import requests

DATA_DIR = Path("data")
ASSETS_DIR = Path("assets")

if not DATA_DIR.exists():
    DATA_DIR = Path(".")

## 2. Phase 1 — Test Google Flights historical price data

The first experiment used a round trip:

- **Origin:** Edmonton (`YEG`)
- **Destination:** Mexico City (`MEX`)
- **Departure:** November 10, 2026
- **Return:** November 20, 2026
- **Currency:** CAD
- **Cabin:** Economy

The SerpApi response contained a `price_insights` object with a `price_history` list.

Each item in `price_history` contains:

1. a Unix timestamp
2. the observed round-trip price

I saved that response as a CSV so the historical analysis can be reproduced without making another API request.

In [ ]:
mex = pd.read_csv(
    DATA_DIR / "yeg_mex_price_history.csv",
    parse_dates=["search_date", "departure_date", "return_date"]
)

mex.head()

## 3. Inspect the historical price dataset

Before plotting anything, I inspect the number of observations, data types, missing values, and basic price statistics.

In [ ]:
mex.shape

In [ ]:
mex.info()

In [ ]:
mex.isna().sum()

In [ ]:
mex["price_cad"].describe()

## 4. Feature engineering — days before departure

The key variable for a booking-timing question is not just the calendar date.

I calculate:

`days_before_departure = departure_date - search_date`

This converts the price history into a feature that can be compared across future trips.

In [ ]:
mex[
    ["search_date", "days_before_departure", "price_cad"]
].head(10)

## 5. Historical price trend: YEG → MEX

I plot price against the number of days remaining before departure.

The x-axis is inverted so the chart reads naturally as the traveler gets closer to departure.

In [ ]:
plt.figure(figsize=(11, 6))

plt.plot(
    mex["days_before_departure"],
    mex["price_cad"],
    marker="o"
)

plt.gca().invert_xaxis()

plt.xlabel("Days before departure")
plt.ylabel("Round-trip price (CAD)")
plt.title("YEG → MEX | Nov 10–20, 2026")

plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
mex.loc[
    mex["price_cad"].idxmin(),
    ["search_date", "days_before_departure", "price_cad"]
]

### Interpretation: YEG → MEX

The saved price history shows that the fare changed substantially during the observation period.

The minimum observed price was **CAD 651**. This is useful evidence that flight prices are dynamic even when the route and travel dates stay fixed.

However, one trip is not enough to conclude that there is a universal "best" number of days before departure. A stronger booking-timing study would repeat this process across many departure dates and routes.

## 6. Phase 2 — Test the idea on Buenos Aires

The next test used:

- **YEG → EZE**
- **November 23–December 5, 2026**

For this run, the exact minimum observation was saved separately.

In [ ]:
eze_min = pd.read_csv(
    DATA_DIR / "yeg_eze_observed_minimum.csv",
    parse_dates=["departure_date", "return_date", "search_date"]
)

eze_min

### Observed minimum

The lowest observed fare in that historical search was:

- **CAD 1,218**
- observed on **July 16, 2026**
- **130 days before departure**

The complete raw EZE price-history response was not saved during that first experiment, so I do not reconstruct or invent those missing values here. The original chart from the analysis is preserved in the `assets` folder.

This is an important data-management lesson: save raw API responses or structured snapshots as soon as they are collected.

![YEG to EZE historical price chart](assets/yeg_eze_price_history_chart.png)

## 7. Phase 3 — Broaden the business question

After testing historical price data, I expanded the project.

Instead of asking only:

> *When should I buy one particular route?*

I also asked:

> **For the same vacation dates, which international destinations are currently the cheapest from Edmonton?**

The fixed travel window was:

- **Departure:** November 23, 2026
- **Return:** December 5, 2026
- **Origin:** Edmonton (`YEG`)
- **Round trip**
- **Economy**
- **1 adult**
- **CAD**

## 8. Build the airport universe

I used the public OurAirports airport dataset as the source for airport metadata.

The search universe included scheduled-service airports in:

- Central America
- South America
- Caribbean countries and territories
- selected Atlantic islands
- Iceland

The first filtered dataset contained **511 airports**.

The code below shows the logic used to create that universe. It requires an internet connection because it reads the current OurAirports CSV.

In [ ]:
country_codes = [
    # Central America
    "BZ", "CR", "SV", "GT", "HN", "NI", "PA",

    # South America
    "AR", "BO", "BR", "CL", "CO", "EC",
    "GY", "PY", "PE", "SR", "UY", "VE",

    # Caribbean / islands
    "AG", "AW", "BS", "BB", "BM", "BQ",
    "VG", "KY", "CU", "CW", "DM", "DO",
    "GD", "GP", "HT", "JM", "MQ", "MS",
    "PR", "BL", "KN", "LC", "MF", "SX",
    "VC", "TT", "TC", "VI",

    # South Atlantic
    "FK",

    # Iceland
    "IS"
]

# Optional live reproduction:
#
# airports = pd.read_csv(
#     "https://raw.githubusercontent.com/davidmegginson/ourairports-data/main/airports.csv"
# )
#
# destinations = airports[
#     (airports["iso_country"].isin(country_codes)) &
#     (airports["scheduled_service"] == "yes") &
#     (airports["iata_code"].notna())
# ].copy()
#
# destinations = destinations[
#     ["iata_code", "name", "municipality", "iso_country", "type"]
# ].drop_duplicates("iata_code")
#
# len(destinations)

### Why I did not query all 511 airports

A 511-airport search would use many API calls and would include a large number of small regional airports that were not useful for the travel decision.

For the next phase I manually selected **30 well-known international or leisure destinations** from the region.

This is a practical project decision, not a statistically derived popularity ranking.

## 9. Selected 30 destinations

In [ ]:
selected_30 = pd.read_csv(DATA_DIR / "selected_30_airports.csv")

selected_30

In [ ]:
selected_30.shape

## 10. Phase 4 — Batch-search the 30 airports

The 30 airport codes were divided into three groups of ten.

SerpApi / Google Flights allows multiple arrival airport codes to be supplied in the same request, which reduces API usage during the first screening step.

The code below reproduces the batch-search logic when `SERPAPI_API_KEY` is available as an environment variable.

In [ ]:
API_KEY = os.getenv("SERPAPI_API_KEY")

codes = selected_30["iata_code"].tolist()

batches = [
    codes[i:i + 10]
    for i in range(0, len(codes), 10)
]

batches

In [ ]:
def search_flight_batch(batch, api_key):
    params = {
        "engine": "google_flights",
        "departure_id": "YEG",
        "arrival_id": ",".join(batch),
        "outbound_date": "2026-11-23",
        "return_date": "2026-12-05",
        "currency": "CAD",
        "hl": "en",
        "gl": "ca",
        "travel_class": "1",
        "type": "1",
        "adults": "1",
        "sort_by": "2",
        "api_key": api_key
    }

    response = requests.get(
        "https://serpapi.com/search.json",
        params=params,
        timeout=60
    )

    return response.json()

## 11. Parse Google Flights results

API responses are nested JSON objects. The main analytical task is to turn that structure into rows and columns.

For each returned itinerary, I extracted:

- destination airport
- price
- airline
- number of outbound stops
- total outbound itinerary duration

In [ ]:
def parse_flights(search_data):
    rows = []

    flights = (
        search_data.get("best_flights", []) +
        search_data.get("other_flights", [])
    )

    for flight in flights:
        legs = flight.get("flights", [])

        if not legs:
            continue

        destination = legs[-1]["arrival_airport"]["id"]

        airlines = sorted({
            leg.get("airline")
            for leg in legs
            if leg.get("airline")
        })

        rows.append({
            "destination": destination,
            "price_cad": flight.get("price"),
            "airline": ", ".join(airlines),
            "outbound_stops": len(legs) - 1,
            "outbound_duration_minutes": flight.get("total_duration")
        })

    return rows

### Optional live collection

The following cell is intentionally disabled by default.

It shows the full collection loop, but a public GitHub notebook should never contain the real API key.

In [ ]:
# all_results = []
#
# if API_KEY:
#     for batch in batches:
#         search_data = search_flight_batch(batch, API_KEY)
#         all_results.extend(parse_flights(search_data))
#
#     results = pd.DataFrame(all_results)
#     results = (
#         results
#         .dropna(subset=["price_cad"])
#         .sort_values("price_cad")
#         .drop_duplicates("destination")
#     )
#
#     results.head(10)
# else:
#     print("SERPAPI_API_KEY is not set. Using saved project snapshots instead.")

## 12. Phase 5 — Verify the cheapest results individually

A batch query is useful for screening, but I did not assume it was sufficient for the final ranking.

I took the 10 cheapest destinations from the batch results and searched each airport individually.

This validation step checked whether the individual search returned the same minimum fare.

In this project, the batch-search and individual-search prices matched for all 10 final destinations.

In [ ]:
verified = pd.read_csv(DATA_DIR / "verified_top10_flights.csv")

verified

## 13. Initial inspection of the verified top 10

Now that the collection and validation stages are complete, I begin the exploratory analysis.

In [ ]:
verified.shape

In [ ]:
verified.info()

In [ ]:
verified.isna().sum()

In [ ]:
verified.describe(include="all")

## 14. Data-quality checks

Even a small dataset should be checked before analysis.

In [ ]:
print("Duplicate destinations:", verified["destination"].duplicated().sum())
print("Non-positive prices:", (verified["price_cad"] <= 0).sum())
print("Negative stop counts:", (verified["outbound_stops"] < 0).sum())
print(
    "Non-positive duration values:",
    (verified["outbound_duration_minutes"] <= 0).sum()
)

## 15. Feature engineering

I convert duration from minutes to hours and create a price rank.

In [ ]:
verified["outbound_duration_hours"] = (
    verified["outbound_duration_minutes"] / 60
).round(1)

verified["price_rank"] = (
    verified["price_cad"]
    .rank(method="dense", ascending=True)
    .astype(int)
)

verified = (
    verified
    .sort_values("price_cad")
    .reset_index(drop=True)
)

verified

## 16. Which destinations were cheapest?

In [ ]:
verified[
    [
        "price_rank",
        "destination",
        "city",
        "country",
        "price_cad",
        "airline",
        "outbound_stops",
        "outbound_duration_hours"
    ]
]

In [ ]:
plt.figure(figsize=(10, 6))

plt.barh(
    verified["city"],
    verified["price_cad"]
)

plt.gca().invert_yaxis()

plt.xlabel("Round-trip price (CAD)")
plt.ylabel("Destination")
plt.title("Verified cheapest destinations from Edmonton")

plt.tight_layout()
plt.show()

### Interpretation: destination ranking

The three cheapest verified destinations were:

1. **Punta Cana — CAD 529**
2. **Montego Bay — CAD 544**
3. **Belize City — CAD 550**

The most expensive destination in the verified top 10 was **Grand Cayman — CAD 738**.

The difference between the first and tenth result was **CAD 209**.

## 17. Price distribution

In [ ]:
plt.figure(figsize=(9, 5))

plt.hist(
    verified["price_cad"],
    bins=6
)

plt.xlabel("Round-trip price (CAD)")
plt.ylabel("Number of destinations")
plt.title("Distribution of verified fares")

plt.tight_layout()
plt.show()

In [ ]:
verified["price_cad"].describe()

### Interpretation: price distribution

The verified fares are concentrated between roughly CAD 500 and CAD 750.

Because this is a selected top-10 shortlist rather than a random sample of all international flights, I do not use the distribution to make broad statistical claims about airfare.

## 18. Premium relative to the cheapest destination

A traveler may find it easier to think in terms of the additional cost compared with the cheapest option.

In [ ]:
cheapest_price = verified["price_cad"].min()

verified["extra_vs_cheapest"] = (
    verified["price_cad"] - cheapest_price
)

verified[
    ["destination", "city", "price_cad", "extra_vs_cheapest"]
]

In [ ]:
plt.figure(figsize=(10, 6))

plt.barh(
    verified["city"],
    verified["extra_vs_cheapest"]
)

plt.gca().invert_yaxis()

plt.xlabel("Extra cost vs cheapest option (CAD)")
plt.ylabel("Destination")
plt.title("Price premium relative to Punta Cana")

plt.tight_layout()
plt.show()

## 19. Stops and price

In [ ]:
stops_summary = (
    verified
    .groupby("outbound_stops")
    .agg(
        destinations=("destination", "count"),
        average_price_cad=("price_cad", "mean"),
        median_price_cad=("price_cad", "median")
    )
    .round(1)
)

stops_summary

In [ ]:
avg_price_by_stops = (
    verified
    .groupby("outbound_stops")["price_cad"]
    .mean()
    .sort_index()
)

plt.figure(figsize=(8, 5))

plt.bar(
    avg_price_by_stops.index.astype(str),
    avg_price_by_stops.values
)

plt.xlabel("Outbound stops")
plt.ylabel("Average round-trip price (CAD)")
plt.title("Average price by number of outbound stops")

plt.tight_layout()
plt.show()

### Interpretation: stops

Punta Cana was both the cheapest result and the only nonstop outbound itinerary in the verified top 10.

That does **not** mean nonstop flights are generally cheaper. The sample was already selected for low prices, and the stop-count groups contain very few observations.

## 20. Price versus outbound itinerary duration

In [ ]:
plt.figure(figsize=(9, 6))

plt.scatter(
    verified["outbound_duration_hours"],
    verified["price_cad"]
)

for _, row in verified.iterrows():
    plt.annotate(
        row["destination"],
        (
            row["outbound_duration_hours"],
            row["price_cad"]
        ),
        xytext=(5, 4),
        textcoords="offset points"
    )

plt.xlabel("Outbound itinerary duration (hours)")
plt.ylabel("Round-trip price (CAD)")
plt.title("Price vs outbound travel time")

plt.tight_layout()
plt.show()

### Interpretation: duration

The cheapest itinerary is not always the shortest, and a low fare can involve a much longer journey.

This is an example of why a travel decision should not rely on price alone. A future version of the project could combine price, duration, and stops into a simple value score.

## 21. Airlines represented in the verified shortlist

In [ ]:
airline_counts = verified["airline"].value_counts()

airline_counts

In [ ]:
plt.figure(figsize=(8, 5))

plt.bar(
    airline_counts.index,
    airline_counts.values
)

plt.xlabel("Airline")
plt.ylabel("Number of destinations")
plt.title("Airlines in the verified top 10")

plt.tight_layout()
plt.show()

### Interpretation: airlines

WestJet appears most frequently in this particular shortlist, followed by United and Air Canada.

This does not measure airline market share and does not prove that one airline is usually cheaper. It describes only these destinations, dates, and search results.

## 22. Key findings

1. Historical price data showed that a fixed itinerary can change meaningfully over time. For YEG → MEX, the lowest observed fare in the saved history was **CAD 651**.
2. For YEG → EZE on November 23–December 5, the observed historical minimum was **CAD 1,218**, recorded **130 days before departure**.
3. Filtering the wider airport universe produced **511 scheduled-service candidate airports** before the project was narrowed to 30 selected destinations.
4. For November 23–December 5, the cheapest individually verified destination was **Punta Cana at CAD 529 round trip**.
5. **Montego Bay (CAD 544)** and **Belize City (CAD 550)** were very close to the cheapest option.
6. The verified top-10 range was **CAD 209** from cheapest to most expensive.
7. Low price did not necessarily imply a short itinerary. Travel time and stops are important decision variables in addition to fare.
8. The project demonstrates two different airfare analytics questions:
   - **booking timing** for a fixed route;
   - **destination comparison** for fixed travel dates.

## 23. Limitations

This is an exploratory junior-analyst project, not a production airfare forecasting system.

Important limitations:

- Airfares are dynamic and can change many times per day.
- The booking-timing analysis uses only a small number of routes and travel periods.
- The destination comparison uses one fixed vacation window.
- The 30-airport shortlist was manually selected from a much larger airport universe; it is not a formal passenger-volume ranking.
- The final EDA uses only the individually verified top 10, creating selection bias.
- The full raw EZE price-history response was not saved during the first experiment.
- Outbound stops and duration describe the returned outbound itinerary associated with the round-trip fare.
- Baggage, fare restrictions, seat selection, and other fees were not included.
- API/search results can change when the analysis is rerun.

## 24. Next steps

A stronger second version of the project would:

- save every raw API response;
- collect prices every day;
- analyze several departure and return periods;
- include both Edmonton (`YEG`) and Calgary (`YYC`);
- calculate `days_before_departure` for every historical search;
- compare median prices across booking windows;
- add return-itinerary duration and stops;
- incorporate baggage or total trip cost;
- visualize the results in Tableau or Power BI;
- consider prediction only after collecting a much larger historical dataset.

## 25. API key security

The real SerpApi key is **not stored in this notebook**.

A public GitHub repository should load credentials from an environment variable:

In [ ]:
API_KEY = os.getenv("SERPAPI_API_KEY")

if API_KEY:
    print("API key found.")
else:
    print("No API key found. Saved CSV snapshots can still be analyzed.")

The repository's `.gitignore` file excludes `.env`, Jupyter checkpoint folders, and Python cache files.

This keeps credentials and temporary files out of the public repository.